# llama2-7b: W-BFP4/A-BiE4 through W-BFP8/A-BiE8 G16 sweep

One Colab run evaluates five matched weight/activation widths. For each width, the notebook
reloads the original FP16 model, collects activation calibration values from WikiText-2 train,
searches independent P95-P99 activation thresholds using that width's BiE reconstruction MSE,
quantizes weights with the matching BFP width, and measures WikiText-2 test PPL.
Each width saves one model-prefixed JSON whose name contains `w-bfpN-a-bieN`. The final cell
downloads one ZIP of the five JSON files. `lm_head` remains FP16.


In [ ]:
%pip install -q "transformers==5.13.1" "datasets==4.0.0" accelerate sentencepiece tqdm

In [ ]:
import gc
import json
import math
import os
import platform
import struct
from functools import lru_cache
import time
import zipfile
from dataclasses import asdict, dataclass, replace
from getpass import getpass
from pathlib import Path

import datasets
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_SLUG = "llama2-7b"
MODEL_ID = "meta-llama/Llama-2-7b-hf"
DATASET_ID = "Salesforce/wikitext"
DATASET_CONFIG = "wikitext-2-raw-v1"
SPLIT = "test"
CALIBRATION_SPLIT = "train"
CALIBRATION_BLOCKS = 8
CALIBRATION_SEED = 0
CONTEXT_LENGTH = 2048
STRIDE = 2048
DROP_REMAINDER = True
EVALUATION_PROTOCOL = "non_overlapping_2048_drop_remainder"
BASELINE_PPL = 5.472103118896484


@dataclass(frozen=True)
class ABiEConfig:
    block_size: int = 16
    shared_exponent_bits: int = 5
    mantissa_bits: int = 4  # Both BFP5 and BiE5: 1 sign bit + 4 magnitude bits.
    rounding: str = "nearest_even"
    threshold_method: str = "independent_percentile_grid_reconstruction_mse"
    candidate_percentiles: tuple = tuple(range(95, 100))
    threshold_exponent_bits: int = 64
    weight_chunk_rows: int = 128
    activation_chunk_rows: int = 256
    activation_histogram_chunk_rows: int = 512
    quantize_lm_head: bool = False

    def validate(self):
        if self.block_size != 16:
            raise ValueError("This experiment requires Group-16.")
        if self.shared_exponent_bits != 5:
            raise ValueError("This experiment requires E5 shared exponents.")
        if self.mantissa_bits <= 0:
            raise ValueError("mantissa_bits must be positive.")
        if self.rounding != "nearest_even":
            raise ValueError("This experiment requires round-to-nearest-even.")
        if self.threshold_method != "independent_percentile_grid_reconstruction_mse":
            raise ValueError("Unexpected threshold method.")
        if self.candidate_percentiles != tuple(range(95, 100)):
            raise ValueError("Expected the inclusive P95-P99 grid at 1-percent steps.")
        if self.threshold_exponent_bits != 64:
            raise ValueError("Threshold log2 values are stored as float64.")
        if min(self.weight_chunk_rows, self.activation_chunk_rows,
               self.activation_histogram_chunk_rows) <= 0:
            raise ValueError("Chunk sizes must be positive.")


ABIE = ABiEConfig()
ABIE.validate()
MANTISSA_BITS_SWEEP = tuple(range(3, 8))  # BFP4/BiE4 through BFP8/BiE8.
PRIVATE_VALUE_BITS = 1 + ABIE.mantissa_bits
OUTPUT_PATH = Path("result") / (
    f"{MODEL_SLUG}-w-bfp{PRIVATE_VALUE_BITS}-a-bie{PRIVATE_VALUE_BITS}-"
    f"g{ABIE.block_size}-p95-p99-grid-mse-rne-fp64exp-"
    "calib-train-no-lm-head-s2048.json"
)
WEIGHT_EFFECTIVE_BITS_PER_VALUE = (
    PRIVATE_VALUE_BITS + ABIE.shared_exponent_bits / ABIE.block_size
)
ACTIVATION_EFFECTIVE_BITS_PER_VALUE = (
    PRIVATE_VALUE_BITS + 1 + 2 * ABIE.shared_exponent_bits / ABIE.block_size
)

torch.manual_seed(0)
torch.backends.cuda.matmul.allow_tf32 = False

print(f"Activation-only BiE sweep: BFP/BiE {[1 + m for m in MANTISSA_BITS_SWEEP]}; G{ABIE.block_size}; P95-P99")
for _mantissa_bits in MANTISSA_BITS_SWEEP:
    _private_bits = 1 + _mantissa_bits
    _weight_bits = _private_bits + ABIE.shared_exponent_bits / ABIE.block_size
    _activation_bits = _private_bits + 1 + 2 * ABIE.shared_exponent_bits / ABIE.block_size
    print(f"BFP{_private_bits}: {_weight_bits:.3f}; BiE{_private_bits}: {_activation_bits:.3f} bits/value (threshold metadata excluded)")
print(f"W-BFP5/A-BiE5 output-name template: {OUTPUT_PATH.resolve()}")
if not torch.cuda.is_available():
    print("CUDA is unavailable: smoke tests can run on CPU; full calibration/PPL cannot.")

## Weight BFP and activation BiE threshold search

For each width, weight tensors use standard single-exponent BFP and have no threshold.
Activation tensors use two shared E5 exponents, one type bit per value, and an independent
threshold selected from exact P95-P99 FP16 calibration values. Candidate MSE uses the
current width's BiE encoder and round-to-nearest-even mantissas.


In [ ]:
COUNT_NAMES = (
    "total_values", "outlier_values", "total_blocks", "outlier_blocks",
    "normal_only_blocks", "outlier_only_blocks", "mixed_blocks",
)


def _empty_counts(device):
    return torch.zeros(len(COUNT_NAMES), dtype=torch.int64, device=device)


@lru_cache(maxsize=None)
def _threshold_value(exponent):
    if not isinstance(exponent, (int, float)) or not math.isfinite(exponent):
        raise ValueError("Threshold log2 exponent must be finite.")
    if exponent == -128.0:
        return 0.0  # Reserved exponent for an all-zero percentile.
    value = math.exp2(float(exponent))
    if value <= 0.0 or value > 65504.0 * (1.0 + 1e-12):
        raise ValueError("Threshold exponent is outside finite FP16 range.")
    return float(struct.unpack("e", struct.pack("e", value))[0])


@torch.no_grad()
def _absolute_fp16_histogram(tensor, chunk_rows):
    width = tensor.shape[-1]
    flat = tensor.reshape(-1, width)
    if flat.numel() == 0 or chunk_rows <= 0:
        raise ValueError("Expected a nonempty tensor and positive chunk_rows.")
    histogram = torch.zeros(32768, dtype=torch.int64, device=tensor.device)
    for start in range(0, flat.size(0), chunk_rows):
        values = flat[start:start + chunk_rows].to(torch.float16).contiguous()
        magnitude_bits = values.view(torch.int16).to(torch.int32).bitwise_and(0x7fff)
        histogram.add_(torch.bincount(magnitude_bits.reshape(-1), minlength=32768))
    if histogram[0x7c00:].sum().item():
        raise ValueError("Threshold search requires finite FP16 values.")
    return histogram


def _percentile_exponent_candidates(histogram, config):
    hist = histogram.detach().cpu()
    total = int(hist.sum().item())
    if total == 0:
        raise ValueError("No values were collected for threshold search.")
    if hist[0x7c00:].sum().item():
        raise ValueError("Threshold search requires finite FP16 values.")
    cumulative = hist.cumsum(0)
    by_value = {}
    for percentile in config.candidate_percentiles:
        rank = math.ceil(percentile * total / 100)
        bit_pattern = int(torch.searchsorted(cumulative, rank).item())
        value = float(torch.tensor([bit_pattern], dtype=torch.int16).view(torch.float16)[0].item())
        if value not in by_value:
            exponent = -128.0 if value == 0.0 else math.log2(value)
            if _threshold_value(exponent) != value:
                raise RuntimeError("Threshold exponent failed FP16 round-trip.")
            by_value[value] = {
                "threshold_exponent": exponent,
                "threshold_value": value,
                "percentiles": [],
            }
        by_value[value]["percentiles"].append(percentile)
    return list(by_value.values())


def _shared_exponent(max_abs, present, config):
    safe_max = max_abs.clamp_min(torch.finfo(torch.float32).tiny)
    exponent = torch.floor(torch.log2(safe_max))
    exp_min = -(1 << (config.shared_exponent_bits - 1))
    exp_max = (1 << (config.shared_exponent_bits - 1)) - 1
    exponent = exponent.clamp(exp_min, exp_max)
    return torch.where(present, exponent, torch.zeros_like(exponent))


@torch.no_grad()
def _quantize_bie_rows_with_threshold(rows, threshold, config):
    original_shape = rows.shape
    width = original_shape[-1]
    flat = rows.reshape(-1, width).float()
    padding = (-width) % config.block_size
    valid = torch.ones_like(flat, dtype=torch.bool)
    if padding:
        flat = F.pad(flat, (0, padding))
        valid = F.pad(valid, (0, padding), value=False)

    padded_width = flat.size(1)
    blocks = flat.reshape(flat.size(0), -1, config.block_size)
    valid_blocks = valid.reshape_as(blocks)
    magnitude = blocks.abs()

    # BiE has one type bit per value; no sub-block alignment is applied.
    outlier = valid_blocks & (magnitude > threshold)
    normal = valid_blocks & ~outlier
    normal_present = normal.any(dim=-1, keepdim=True)
    outlier_present = outlier.any(dim=-1, keepdim=True)
    normal_max = torch.where(normal, magnitude, 0.0).amax(dim=-1, keepdim=True)
    outlier_max = torch.where(outlier, magnitude, 0.0).amax(dim=-1, keepdim=True)
    normal_exp = _shared_exponent(normal_max, normal_present, config)
    outlier_exp = _shared_exponent(outlier_max, outlier_present, config)
    normal_exp = torch.where(~normal_present & outlier_present, outlier_exp, normal_exp)
    outlier_exp = torch.where(~outlier_present & normal_present, normal_exp, outlier_exp)
    selected_exp = torch.where(outlier, outlier_exp, normal_exp)

    step = torch.pow(2.0, selected_exp - (config.mantissa_bits - 1))
    mantissa_max = (1 << config.mantissa_bits) - 1
    mantissa = torch.round(blocks / step).clamp(-mantissa_max, mantissa_max)
    dequantized = (mantissa * step).reshape(flat.size(0), padded_width)
    dequantized = dequantized[:, :width].reshape(original_shape).to(rows.dtype)

    real_block = valid_blocks.any(dim=-1, keepdim=True)
    normal_only = real_block & normal_present & ~outlier_present
    outlier_only = real_block & outlier_present & ~normal_present
    mixed = real_block & normal_present & outlier_present
    counts = torch.stack((
        torch.tensor(rows.numel(), dtype=torch.int64, device=rows.device),
        outlier.sum(dtype=torch.int64), real_block.sum(dtype=torch.int64),
        outlier_present.sum(dtype=torch.int64), normal_only.sum(dtype=torch.int64),
        outlier_only.sum(dtype=torch.int64), mixed.sum(dtype=torch.int64),
    ))
    return dequantized, counts


@torch.no_grad()
def quantize_bie(tensor, config, chunk_rows, threshold_exponent):
    width = tensor.shape[-1]
    flat = tensor.reshape(-1, width)
    if flat.numel() == 0 or chunk_rows <= 0:
        raise ValueError("Expected a nonempty tensor and positive chunk_rows.")
    threshold = _threshold_value(threshold_exponent)
    output = torch.empty_like(flat)
    counts = _empty_counts(flat.device)
    for start in range(0, flat.size(0), chunk_rows):
        end = min(start + chunk_rows, flat.size(0))
        quantized, chunk_counts = _quantize_bie_rows_with_threshold(
            flat[start:end], threshold, config
        )
        output[start:end] = quantized
        counts.add_(chunk_counts)
    return output.reshape_as(tensor), counts


@torch.no_grad()
def _reconstruction_sse(tensor, config, chunk_rows, threshold_exponent):
    width = tensor.shape[-1]
    flat = tensor.reshape(-1, width)
    total = torch.zeros((), dtype=torch.float64, device=tensor.device)
    threshold = _threshold_value(threshold_exponent)
    for start in range(0, flat.size(0), chunk_rows):
        original = flat[start:start + chunk_rows]
        quantized, _ = _quantize_bie_rows_with_threshold(original, threshold, config)
        total.add_((original.float() - quantized.float()).square().sum(dtype=torch.float64))
    return total


@torch.no_grad()
def _quantize_weight_bfp_rows(rows, config):
    original_shape = rows.shape
    width = original_shape[-1]
    flat = rows.reshape(-1, width).float()
    padding = (-width) % config.block_size
    if padding:
        flat = F.pad(flat, (0, padding))
    padded_width = flat.size(1)
    blocks = flat.reshape(flat.size(0), -1, config.block_size)
    max_abs = blocks.abs().amax(dim=-1, keepdim=True)
    exponent = _shared_exponent(max_abs, max_abs != 0, config)
    step = torch.pow(2.0, exponent - (config.mantissa_bits - 1))
    mantissa_max = (1 << config.mantissa_bits) - 1
    mantissa = torch.round(blocks / step).clamp(-mantissa_max, mantissa_max)
    output = (mantissa * step).reshape(flat.size(0), padded_width)
    return output[:, :width].reshape(original_shape).to(rows.dtype)


@torch.no_grad()
def quantize_weight_in_place(weight, config):
    total_values = 0
    total_blocks = 0
    blocks_per_row = math.ceil(weight.size(-1) / config.block_size)
    for start in range(0, weight.size(0), config.weight_chunk_rows):
        end = min(start + config.weight_chunk_rows, weight.size(0))
        rows = weight[start:end]
        rows.copy_(_quantize_weight_bfp_rows(rows, config))
        total_values += rows.numel()
        total_blocks += rows.size(0) * blocks_per_row
    return {"total_values": total_values, "total_blocks": total_blocks}




## Activation calibration and Linear replacement

The original FP16 model supplies activation values for percentile candidates and reconstruction
MSE. After thresholds are fixed, weight tensors are quantized once with BFP5 and activations
are quantized with BiE5 on each inference call.


In [ ]:
def _rate(numerator, denominator):
    return {
        "numerator": int(numerator),
        "denominator": int(denominator),
        "rate": None if denominator == 0 else numerator / denominator,
    }


def _counts_to_dict(counts):
    values = counts.detach().cpu().tolist()
    return {name: int(value) for name, value in zip(COUNT_NAMES, values)}


def _summarize_counts(counts):
    return {
        "counts": counts,
        "rates": {
            "outlier_value_rate": _rate(counts["outlier_values"], counts["total_values"]),
            "secondary_exponent_value_rate": _rate(counts["outlier_values"], counts["total_values"]),
            "outlier_block_rate": _rate(counts["outlier_blocks"], counts["total_blocks"]),
            "normal_only_block_rate": _rate(counts["normal_only_blocks"], counts["total_blocks"]),
            "outlier_only_block_rate": _rate(counts["outlier_only_blocks"], counts["total_blocks"]),
            "mixed_block_rate": _rate(counts["mixed_blocks"], counts["total_blocks"]),
        },
    }


def _quantized_linear_modules(model, config):
    return {
        name: module for name, module in model.named_modules()
        if isinstance(module, nn.Linear)
        and (config.quantize_lm_head or name != "lm_head")
    }


@torch.inference_mode()
def _run_calibration_forward(model, batches, label):
    device = next(model.parameters()).device
    for batch in tqdm(batches, desc=label):
        output = model(batch.to(device), use_cache=False)
        del output


@torch.inference_mode()
def collect_activation_histograms(model, batches, config):
    modules = _quantized_linear_modules(model, config)
    histograms = {
        name: torch.zeros(32768, dtype=torch.int64, device=module.weight.device)
        for name, module in modules.items()
    }
    hooks = []

    def make_hook(name):
        def hook(_module, inputs):
            histogram = _absolute_fp16_histogram(
                inputs[0], config.activation_histogram_chunk_rows
            )
            histograms[name].add_(histogram)
        return hook

    try:
        for name, module in modules.items():
            hooks.append(module.register_forward_pre_hook(make_hook(name)))
        _run_calibration_forward(model, batches, "Activation percentile histograms")
    finally:
        for hook in hooks:
            hook.remove()

    for name, histogram in histograms.items():
        if not histogram.sum().item():
            raise RuntimeError(f"No calibration activations for {name}.")
    return histograms


@torch.inference_mode()
def search_activation_thresholds(model, batches, histograms, config):
    modules = _quantized_linear_modules(model, config)
    if set(modules) != set(histograms):
        raise RuntimeError("Activation histogram modules do not match the model.")
    candidates = {
        name: _percentile_exponent_candidates(histograms[name], config)
        for name in modules
    }
    sse = {
        name: torch.zeros(len(candidates[name]), dtype=torch.float64, device=module.weight.device)
        for name, module in modules.items()
    }
    counts = {name: 0 for name in modules}
    hooks = []

    def make_hook(name):
        def hook(_module, inputs):
            x = inputs[0].detach()
            width = x.shape[-1]
            flat = x.reshape(-1, width)
            counts[name] += x.numel()
            for index, candidate in enumerate(candidates[name]):
                error = _reconstruction_sse(
                    flat, config, config.activation_chunk_rows,
                    candidate["threshold_exponent"],
                )
                sse[name][index].add_(error)
        return hook

    try:
        for name, module in modules.items():
            hooks.append(module.register_forward_pre_hook(make_hook(name)))
        _run_calibration_forward(model, batches, "Activation threshold MSE")
    finally:
        for hook in hooks:
            hook.remove()

    selected = {}
    diagnostics = {}
    for name in modules:
        if counts[name] == 0:
            raise RuntimeError(f"No activation MSE samples for {name}.")
        for index, candidate in enumerate(candidates[name]):
            candidate["mse"] = float((sse[name][index] / counts[name]).item())
        best = min(
            candidates[name],
            key=lambda row: (row["mse"], row["threshold_value"]),
        )
        selected[name] = best["threshold_exponent"]
        diagnostics[name] = {
            "calibration_values": counts[name],
            "percentile_grid": list(config.candidate_percentiles),
            "candidate_thresholds": candidates[name],
            "selected_exponent": best["threshold_exponent"],
            "selected_mse": best["mse"],
            "selection_objective": "tensor_reconstruction_mse",
        }
    return selected, diagnostics


class ABiELinear(nn.Module):
    def __init__(self, linear, config, weight_counts, activation_exponent, activation_search):
        super().__init__()
        self.linear = linear
        self.config = config
        self.weight_counts = weight_counts
        self.activation_threshold_exponent = float(activation_exponent)
        self.activation_search = activation_search
        device = linear.weight.device
        self.register_buffer("_activation_counts", _empty_counts(device), persistent=False)
        self.register_buffer(
            "_activation_calls", torch.zeros((), dtype=torch.int64, device=device),
            persistent=False,
        )

    def forward(self, x):
        x_bie, counts = quantize_bie(
            x, self.config, self.config.activation_chunk_rows,
            self.activation_threshold_exponent,
        )
        self._activation_counts.add_(counts)
        self._activation_calls.add_(1)
        return F.linear(x_bie, self.linear.weight, self.linear.bias).to(torch.float16)

    def export_stats(self):
        return {
            "weight": {"format": "BFP", "counts": self.weight_counts},
            "activation": {
                "format": "BiE",
                "threshold_exponent": self.activation_threshold_exponent,
                "threshold_value": _threshold_value(self.activation_threshold_exponent),
                "search": self.activation_search,
                "inference_calls": int(self._activation_calls.detach().cpu().item()),
                **_summarize_counts(_counts_to_dict(self._activation_counts)),
            },
        }


@torch.no_grad()
def replace_linear_layers(model, config, activation_exponents, activation_search):
    modules = _quantized_linear_modules(model, config)
    if set(modules) != set(activation_exponents) or set(modules) != set(activation_search):
        raise RuntimeError("Activation thresholds must cover every quantized Linear.")
    replaced = {}
    for name, linear in tqdm(modules.items(), desc="BFP weight quantization"):
        weight_counts = quantize_weight_in_place(linear.weight, config)
        wrapper = ABiELinear(
            linear, config, weight_counts,
            activation_exponents[name], activation_search[name],
        )
        parent_name, _, child_name = name.rpartition(".")
        parent = model.get_submodule(parent_name) if parent_name else model
        setattr(parent, child_name, wrapper)
        replaced[name] = wrapper
    return replaced


def _add_counts(total, current):
    for name in COUNT_NAMES:
        total[name] += current[name]


def _exponent_summary(values):
    return {
        "count": len(values),
        "min": min(values) if values else None,
        "max": max(values) if values else None,
        "mean": sum(values) / len(values) if values else None,
    }


def export_quantization_stats(replaced):
    weight_totals = {"total_values": 0, "total_blocks": 0}
    activation_totals = {name: 0 for name in COUNT_NAMES}
    activation_exponents = []
    layers = []
    for name in sorted(replaced):
        layer_stats = replaced[name].export_stats()
        layers.append({"layer_name": name, **layer_stats})
        for key in weight_totals:
            weight_totals[key] += layer_stats["weight"]["counts"][key]
        _add_counts(activation_totals, layer_stats["activation"]["counts"])
        activation_exponents.append(layer_stats["activation"]["threshold_exponent"])
    return {
        "aggregate": {
            "weight": {"format": "BFP", "counts": weight_totals},
            "activation": {
                "format": "BiE",
                "threshold_exponent_summary": _exponent_summary(activation_exponents),
                **_summarize_counts(activation_totals),
            },
        },
        "layers": layers,
    }


## Synthetic checks

Check BFP weights, BiE activation quantization, exact percentile candidates, independent
calibration thresholds, and the weight/activation statistics schema.


In [ ]:
_test_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

_zero = torch.zeros((2, 16), dtype=torch.float32, device=_test_device)
_zero_candidates = _percentile_exponent_candidates(
    _absolute_fp16_histogram(_zero, 1), ABIE
)
assert len(_zero_candidates) == 1
assert _zero_candidates[0]["threshold_exponent"] == -128.0
_zero_q, _zero_counts = quantize_bie(_zero, ABIE, 1, -128.0)
assert torch.equal(_zero_q, _zero)
assert _counts_to_dict(_zero_counts)["outlier_values"] == 0

_dense = torch.linspace(0.6, 0.9, 64, device=_test_device).reshape(4, 16)
_dense_candidates = _percentile_exponent_candidates(
    _absolute_fp16_histogram(_dense, 2), ABIE
)
assert 2 <= len(_dense_candidates) <= len(ABIE.candidate_percentiles)
assert all(0.6 <= row["threshold_value"] <= 0.91 for row in _dense_candidates)
assert all(row["threshold_value"] < 1.0 for row in _dense_candidates)

_mixed = torch.zeros((1, 16), dtype=torch.float32, device=_test_device)
_mixed[0, :4] = torch.tensor([0.9, -0.4, 4.5, 0.84375], device=_test_device)
_mixed_q, _mixed_counts = quantize_bie(_mixed, ABIE, 1, 0)
assert _counts_to_dict(_mixed_counts)["outlier_values"] == 1
assert _mixed_q[0, 3].item() == 0.875  # Round to nearest even.

_weight = torch.tensor([[0.5, 0.75] + [0.0] * 14], device=_test_device)
_weight_q = _quantize_weight_bfp_rows(_weight, ABIE)
assert _weight_q[0, 0].item() == 0.5
assert _weight_q[0, 1].item() == 0.75
_weight_counts = quantize_weight_in_place(_weight, ABIE)
assert _weight_counts == {"total_values": 16, "total_blocks": 1}

class _ToyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Linear(16, 8, bias=False, device=_test_device)
        self.proj2 = nn.Linear(16, 8, bias=False, device=_test_device)
        with torch.no_grad():
            self.proj2.weight.copy_(self.proj.weight * 16)

    def forward(self, x, use_cache=False):
        return self.proj(x) + self.proj2(x * 8)


_toy = _ToyModel().eval()
_calib_batches = [
    torch.linspace(-2, 2, 32, device=_test_device).reshape(2, 16),
    torch.linspace(-0.5, 0.5, 48, device=_test_device).reshape(3, 16),
]
_activation_hist = collect_activation_histograms(_toy, _calib_batches, ABIE)
_activation_exp, _activation_search = search_activation_thresholds(
    _toy, _calib_batches, _activation_hist, ABIE
)
assert set(_activation_exp) == {"proj", "proj2"}
assert _activation_search["proj"]["calibration_values"] == 80
assert math.isclose(_activation_exp["proj2"], _activation_exp["proj"] + 3, abs_tol=1e-12)
_raw_activation = torch.cat(_calib_batches, dim=0)
for _candidate in _activation_search["proj"]["candidate_thresholds"]:
    _q, _ = quantize_bie(_raw_activation, ABIE, 1, _candidate["threshold_exponent"])
    _mse = (_raw_activation - _q).square().mean().item()
    assert math.isclose(_candidate["mse"], _mse, rel_tol=1e-5, abs_tol=1e-8)

_original_weight = _toy.proj.weight.detach().clone()
_toy_layers = replace_linear_layers(_toy, ABIE, _activation_exp, _activation_search)
assert len(_toy_layers) == 2
assert torch.equal(_toy_layers["proj"].linear.weight, _quantize_weight_bfp_rows(_original_weight, ABIE))
_toy_y = _toy(_calib_batches[0])
_toy_stats = _toy_layers["proj"].export_stats()
assert _toy_y.shape == (2, 8)
assert torch.isfinite(_toy_y).all()
assert "threshold_exponent" not in _toy_stats["weight"]
assert _toy_stats["activation"]["inference_calls"] == 1
assert _toy_stats["activation"]["threshold_exponent"] == _activation_exp["proj"]
_toy_aggregate = export_quantization_stats(_toy_layers)["aggregate"]
assert _toy_aggregate["weight"]["counts"]["total_blocks"] == 16
assert _toy_aggregate["activation"]["counts"]["total_values"] == 64
assert json.loads(json.dumps(_toy_stats))["weight"]["format"] == "BFP"

del _zero, _dense, _mixed, _weight, _toy, _toy_y
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
for _private_bits in range(4, 9):
    _config = replace(ABIE, mantissa_bits=_private_bits - 1)
    _config.validate()
    _check_rows = torch.tensor([[0.25, -0.4, 0.9, 4.5] + [0.0] * 12], device=_test_device)
    _weight_q = _quantize_weight_bfp_rows(_check_rows, _config)
    _activation_q, _activation_counts = quantize_bie(_check_rows, _config, 1, 0.0)
    assert _weight_q.shape == _activation_q.shape == _check_rows.shape
    assert _counts_to_dict(_activation_counts)["outlier_values"] == 1
print("Synthetic W-BFP4/A-BiE4 through W-BFP8/A-BiE8 checks passed.")


## Dataset and tokenizer

The train split provides calibration tensors. The test split remains exclusive to PPL evaluation.
Calibration draws a fixed number of non-overlapping 2048-token blocks with a recorded seed.

In [ ]:
token = os.getenv("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
if not token:
    token = getpass("HF_TOKEN: ")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=token)
test_dataset = load_dataset(DATASET_ID, DATASET_CONFIG, split=SPLIT)
test_text = "\n\n".join(test_dataset["text"])
input_ids = tokenizer(test_text, return_tensors="pt").input_ids

calibration_dataset = load_dataset(DATASET_ID, DATASET_CONFIG, split=CALIBRATION_SPLIT)
calibration_text = "\n\n".join(calibration_dataset["text"])
calibration_ids = tokenizer(calibration_text, return_tensors="pt").input_ids
calibration_source_tokens = calibration_ids.numel()
available_calibration_blocks = calibration_source_tokens // CONTEXT_LENGTH
if available_calibration_blocks < CALIBRATION_BLOCKS:
    raise RuntimeError("Not enough full train blocks for calibration.")
generator = torch.Generator().manual_seed(CALIBRATION_SEED)
calibration_block_indices = sorted(
    torch.randperm(available_calibration_blocks, generator=generator)[:CALIBRATION_BLOCKS].tolist()
)
calibration_batches = [
    calibration_ids[:, index * CONTEXT_LENGTH:(index + 1) * CONTEXT_LENGTH].clone()
    for index in calibration_block_indices
]
del test_dataset, test_text, calibration_dataset, calibration_text, calibration_ids
print(f"WikiText-2 test tokens: {input_ids.numel():,}")
print(f"Calibration: {len(calibration_batches)} train blocks, seed {CALIBRATION_SEED}")

## Non-overlapping perplexity evaluation

In [ ]:
@torch.inference_mode()
def evaluate_perplexity(model, input_ids, context_length, stride, drop_remainder):
    if stride != context_length:
        raise ValueError("Non-overlapping evaluation requires stride == context_length.")
    if not drop_remainder:
        raise ValueError("This evaluation requires drop_remainder=True.")
    if context_length > model.config.max_position_embeddings:
        raise ValueError("context_length exceeds the model context window.")

    device = next(model.parameters()).device
    sequence_length = input_ids.size(1)
    usable_length = sequence_length // context_length * context_length
    dropped_tokens = sequence_length - usable_length
    if usable_length == 0:
        raise ValueError("Input does not contain a complete context block.")

    total_nll = 0.0
    total_loss_tokens = 0
    total_blocks = usable_length // context_length

    torch.cuda.reset_peak_memory_stats(device)
    torch.cuda.synchronize(device)
    start_time = time.perf_counter()

    for begin in tqdm(
        range(0, usable_length, stride),
        total=total_blocks,
        desc=f"Evaluating BiE{PRIVATE_VALUE_BITS} G{ABIE.block_size}",
    ):
        end = begin + context_length
        batch = input_ids[:, begin:end].to(device)
        labels = batch.clone()
        loss = model(batch, labels=labels, use_cache=False).loss
        loss_tokens = labels[:, 1:].numel()
        total_nll += loss.float().item() * loss_tokens
        total_loss_tokens += loss_tokens

    torch.cuda.synchronize(device)
    elapsed_seconds = time.perf_counter() - start_time
    mean_nll = total_nll / total_loss_tokens

    return {
        "mean_nll": mean_nll,
        "perplexity": float(torch.exp(torch.tensor(mean_nll))),
        "source_input_tokens": sequence_length,
        "used_input_tokens": usable_length,
        "dropped_input_tokens": dropped_tokens,
        "evaluated_blocks": total_blocks,
        "evaluated_tokens": total_loss_tokens,
        "elapsed_seconds": elapsed_seconds,
        "tokens_per_second": total_loss_tokens / elapsed_seconds,
        "peak_gpu_memory_gib": (
            torch.cuda.max_memory_allocated(device) / 2**30
        ),
    }

## Fresh FP16 model and activation profiling for each width

Each sweep iteration reloads FP16 weights and calibrates activations before quantizing weights.
The selected threshold is fixed for inference. The profiling pass records the exact number of
activation outliers per G16 block on the same test tokens used for PPL.


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Full calibration and PPL evaluation require an NVIDIA CUDA GPU.")

def load_and_quantize_model(config):
    if not torch.cuda.is_available():
        raise RuntimeError("Full calibration and PPL evaluation require an NVIDIA CUDA GPU.")

    torch.manual_seed(0)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.float16,
        device_map=0,
        attn_implementation="eager",
        token=token,
    )
    model.eval()
    model.config.use_cache = False

    original_linear_modules = _quantized_linear_modules(model, config)
    if len(original_linear_modules) != 224:
        raise RuntimeError(
            f"Expected 224 decoder Linear layers, got {len(original_linear_modules)}."
        )

    activation_histograms = collect_activation_histograms(model, calibration_batches, config)
    activation_exponents, activation_search = search_activation_thresholds(
        model, calibration_batches, activation_histograms, config
    )
    del activation_histograms
    torch.cuda.empty_cache()

    abie_layers = replace_linear_layers(model, config, activation_exponents, activation_search)
    if set(abie_layers) != set(original_linear_modules):
        raise RuntimeError("Quantized decoder Linear layer set changed.")
    if "lm_head" in abie_layers or not isinstance(model.lm_head, nn.Linear):
        raise RuntimeError("lm_head must remain an FP16 nn.Linear.")
    del original_linear_modules, activation_exponents, activation_search
    torch.cuda.empty_cache()

    parameter_dtypes = {
        parameter.dtype for parameter in model.parameters() if parameter.is_floating_point()
    }
    parameter_devices = {parameter.device.type for parameter in model.parameters()}
    assert parameter_dtypes == {torch.float16}, parameter_dtypes
    assert parameter_devices == {"cuda"}, parameter_devices

    print(f"W-BFP{1 + config.mantissa_bits}/A-BiE{1 + config.mantissa_bits} quantized Linear layers: {len(abie_layers)}")
    print("lm_head: FP16; activation threshold: per-tensor P95-P99 exponent grid, reconstruction MSE")
    print("Format: W-BFP5 (one E5), A-BiE5 (type bit, two E5), G16, nearest-even")
    return model, abie_layers


@torch.inference_mode()
def profile_activation_outliers_per_block(model, input_ids, layers, config, result):
    device = next(model.parameters()).device
    histograms = {
        name: torch.zeros(config.block_size + 1, dtype=torch.int64, device=device)
        for name in layers
    }
    saved_counters = {
        name: (layer._activation_counts.clone(), layer._activation_calls.clone())
        for name, layer in layers.items()
    }
    hooks = []

    def make_hook(name, layer):
        threshold = _threshold_value(layer.activation_threshold_exponent)

        def hook(_module, inputs):
            x = inputs[0].detach()
            width = x.shape[-1]
            flat = x.reshape(-1, width)
            padding = (-width) % config.block_size
            for start in range(0, flat.size(0), config.activation_chunk_rows):
                values = flat[start:start + config.activation_chunk_rows].float()
                valid = torch.ones_like(values, dtype=torch.bool)
                if padding:
                    values = F.pad(values, (0, padding))
                    valid = F.pad(valid, (0, padding), value=False)
                outlier = values.abs().gt(threshold) & valid
                per_block = outlier.reshape(-1, config.block_size).sum(dim=-1)
                histograms[name].add_(
                    torch.bincount(per_block.reshape(-1), minlength=config.block_size + 1)
                )
        return hook

    usable_length = input_ids.size(1) // CONTEXT_LENGTH * CONTEXT_LENGTH
    if STRIDE != CONTEXT_LENGTH or not DROP_REMAINDER or usable_length == 0:
        raise RuntimeError("Occupancy profiling requires the existing non-overlapping protocol.")
    try:
        for name, layer in layers.items():
            hooks.append(layer.register_forward_pre_hook(make_hook(name, layer)))
        for begin in tqdm(
            range(0, usable_length, STRIDE),
            total=usable_length // STRIDE,
            desc="Profiling activation outliers per G16 block",
        ):
            batch = input_ids[:, begin:begin + CONTEXT_LENGTH].to(device)
            output = model(batch, use_cache=False)
            del output
    finally:
        for hook in hooks:
            hook.remove()
        for name, layer in layers.items():
            counts, calls = saved_counters[name]
            layer._activation_counts.copy_(counts)
            layer._activation_calls.copy_(calls)

    aggregate = [0] * (config.block_size + 1)
    per_layer = []
    original_layers = {
        row["layer_name"]: row["activation"]["counts"]
        for row in result["quantization_stats"]["layers"]
    }
    for name in sorted(layers):
        hist = [int(value) for value in histograms[name].cpu().tolist()]
        expected = original_layers[name]
        if sum(hist) != expected["total_blocks"]:
            raise RuntimeError(f"Block histogram total differs for {name}.")
        if sum(k * count for k, count in enumerate(hist)) != expected["outlier_values"]:
            raise RuntimeError(f"Outlier histogram total differs for {name}.")
        if hist[0] != expected["normal_only_blocks"]:
            raise RuntimeError(f"Zero-outlier block count differs for {name}.")
        per_layer.append({"layer_name": name, "outliers_per_block_histogram": hist})
        aggregate = [a + b for a, b in zip(aggregate, hist)]

    expected = result["quantization_stats"]["aggregate"]["activation"]["counts"]
    total_blocks = sum(aggregate)
    total_outliers = sum(k * count for k, count in enumerate(aggregate))
    if total_blocks != expected["total_blocks"]:
        raise RuntimeError("Aggregate histogram block count differs from PPL evaluation.")
    if total_outliers != expected["outlier_values"]:
        raise RuntimeError("Aggregate histogram outlier count differs from PPL evaluation.")
    if aggregate[0] != expected["normal_only_blocks"]:
        raise RuntimeError("Aggregate zero-outlier block count differs from PPL evaluation.")
    if aggregate[config.block_size] != expected["outlier_only_blocks"]:
        raise RuntimeError("Aggregate all-outlier block count differs from PPL evaluation.")
    affected_blocks = total_blocks - aggregate[0]
    return {
        "histogram_semantics": "index k = G16 activation blocks with exactly k outlier values",
        "outliers_per_block_histogram": aggregate,
        "fraction_of_all_blocks": [count / total_blocks for count in aggregate],
        "mean_outliers_per_all_blocks": total_outliers / total_blocks,
        "mean_outliers_per_affected_block": total_outliers / affected_blocks if affected_blocks else 0.0,
        "per_layer_histograms": per_layer,
        "evaluated_blocks": usable_length // CONTEXT_LENGTH,
        "evaluated_tokens": result["evaluated_tokens"],
    }



## Evaluate all five matched widths

The loop recalculates each activation threshold from the FP16 model at the current mantissa
width, then evaluates PPL and saves the activation outlier-per-block histogram to its JSON.
Results from one width do not feed into the next.


In [ ]:
def evaluate_and_save_width(model, layers, config, output_path):
    private_value_bits = 1 + config.mantissa_bits
    weight_effective_bits_per_value = (
        private_value_bits + config.shared_exponent_bits / config.block_size
    )
    activation_effective_bits_per_value = (
        private_value_bits + 1 + 2 * config.shared_exponent_bits / config.block_size
    )
    metrics = evaluate_perplexity(
        model, input_ids, CONTEXT_LENGTH, STRIDE, DROP_REMAINDER,
    )
    quantization_stats = export_quantization_stats(layers)
    weight_counts = quantization_stats["aggregate"]["weight"]["counts"]
    activation_counts = quantization_stats["aggregate"]["activation"]["counts"]
    if weight_counts["total_values"] <= 0 or weight_counts["total_blocks"] <= 0:
        raise RuntimeError("Weight BFP counts are empty.")
    if activation_counts["total_blocks"] != (
        activation_counts["normal_only_blocks"]
        + activation_counts["outlier_only_blocks"]
        + activation_counts["mixed_blocks"]
    ):
        raise RuntimeError("Activation block partition counts are inconsistent.")
    if activation_counts["outlier_values"] > activation_counts["total_values"]:
        raise RuntimeError("Activation outlier count exceeds total values.")
    if activation_counts["outlier_blocks"] != (
        activation_counts["outlier_only_blocks"] + activation_counts["mixed_blocks"]
    ):
        raise RuntimeError("Activation outlier-block counts are inconsistent.")

    threshold_exponents = {
        name: layer.activation_threshold_exponent
        for name, layer in sorted(layers.items())
    }
    if len(threshold_exponents) != len(layers):
        raise RuntimeError("Activation threshold map is incomplete.")
    for layer_stats in quantization_stats["layers"]:
        data = layer_stats["activation"]
        candidates = data["search"]["candidate_thresholds"]
        percentile_map = {
            percentile: candidate["threshold_value"]
            for candidate in candidates
            for percentile in candidate["percentiles"]
        }
        if sorted(percentile_map) != list(config.candidate_percentiles):
            raise RuntimeError("Activation percentile grid is incomplete.")
        p95, p99 = percentile_map[95], percentile_map[99]
        if not all(p95 <= candidate["threshold_value"] <= p99 for candidate in candidates):
            raise RuntimeError("Activation threshold fell outside P95-P99.")
        best = min(candidates, key=lambda row: (row["mse"], row["threshold_value"]))
        if data["threshold_exponent"] != best["threshold_exponent"]:
            raise RuntimeError("Selected activation threshold does not minimize reconstruction MSE.")
        if _threshold_value(best["threshold_exponent"]) != best["threshold_value"]:
            raise RuntimeError("Selected activation threshold exponent failed FP16 round-trip.")

    result = {
        "model": MODEL_ID,
        "dataset": f"{DATASET_ID}/{DATASET_CONFIG}",
        "split": SPLIT,
        "quantization": (
            f"W-BFP{private_value_bits} / A-BiE{private_value_bits} G16 fake quantization "
            "with independent per-activation-tensor percentile-grid reconstruction-MSE thresholds"
        ),
        "method_label": "Activation-only BiE tensor-wise exact P95-P99 threshold grid search",
        "format": (
            f"W-BFP{private_value_bits} (1S{config.mantissa_bits}M + one E{config.shared_exponent_bits}, G{config.block_size}); "
            f"A-BiE{private_value_bits} (1S{config.mantissa_bits}M + two E{config.shared_exponent_bits} "
            f"+ per-value type bit, G{config.block_size})"
        ),
        "abie_config": asdict(config),
        "threshold_contract": {
            "applies_to": "activation only; weights use single-exponent BFP",
            "candidate_distribution": "absolute FP16 calibration activation values",
            "percentile_method": "nearest_rank",
            "percentiles": list(config.candidate_percentiles),
            "exponent_mapping": "float64 log2(exact FP16 percentile value); zero uses -128.0 sentinel",
            "candidate_deduplication": "identical FP16 percentile values only",
            "comparison": "outlier iff abs(x) > FP16_round(2 ** threshold_exponent)",
            "selection": "independent argmin activation tensor reconstruction MSE",
            "activation_granularity": "one nn.Linear input tensor across all calibration calls",
            "activation_threshold_frozen_for_inference": True,
            "threshold_storage": "float64 log2 exponent per activation tensor",
            "computed_before_chunking": True,
        },
        "calibration": {
            "dataset": f"{DATASET_ID}/{DATASET_CONFIG}",
            "split": CALIBRATION_SPLIT,
            "source_tokens": calibration_source_tokens,
            "context_length": CONTEXT_LENGTH,
            "sampled_blocks": CALIBRATION_BLOCKS,
            "sampled_tokens": CALIBRATION_BLOCKS * CONTEXT_LENGTH,
            "seed": CALIBRATION_SEED,
            "block_indices": calibration_block_indices,
            "activation_source": "unquantized FP16 model Linear inputs",
        },
        "storage_contract": {
            "weight_private_value_bits": private_value_bits,
            "weight_shared_exponents_per_block": 1,
            "weight_type_bits_per_value": 0,
            "activation_private_value_bits": private_value_bits,
            "activation_type_bits_per_value": 1,
            "activation_shared_exponents_per_block": 2,
            "shared_exponent_bits": config.shared_exponent_bits,
            "weight_effective_bits_per_value": weight_effective_bits_per_value,
            "activation_effective_bits_per_value_excluding_tensor_threshold": activation_effective_bits_per_value,
            "mantissa_rounding_contract": "round-to-nearest-even (torch.round), then symmetric saturation",
            "threshold_exponent_bits_per_activation_tensor": config.threshold_exponent_bits,
            "packed_storage_implemented": False,
        },
        "quantized_scope": {
            "operations": "all decoder nn.Linear modules",
            "weights": "single-exponent BFP",
            "activations": "dual-exponent BiE",
            "lm_head": config.quantize_lm_head,
            "attention_internal_matmul": False,
            "softmax": False,
            "layernorm": False,
        },
        "linear_output_dtype": "float16",
        "matmul_backend": "torch.nn.functional.linear with dequantized FP16 operands",
        "quantized_linear_layers": len(layers),
        "attention_implementation": "eager",
        "context_length": CONTEXT_LENGTH,
        "stride": STRIDE,
        "evaluation_protocol": EVALUATION_PROTOCOL,
        "drop_remainder": DROP_REMAINDER,
        "baseline_perplexity": BASELINE_PPL,
        "delta_perplexity": metrics["perplexity"] - BASELINE_PPL,
        "threshold_exponents": {"activation": threshold_exponents},
        "quantization_stats": quantization_stats,
        "gpu": torch.cuda.get_device_name(0),
        "cuda": torch.version.cuda,
        "python": platform.python_version(),
        "pytorch": torch.__version__,
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
        **metrics,
    }

    result["activation_block_outlier_profile"] = profile_activation_outliers_per_block(
        model, input_ids, layers, config, result
    )
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
    print(json.dumps({
        "format": f"W-BFP{private_value_bits}/A-BiE{private_value_bits}",
        "perplexity": result["perplexity"],
        "delta_perplexity": result["delta_perplexity"],
        "activation_outlier_rate": result["quantization_stats"]["aggregate"]["activation"]["rates"]["outlier_value_rate"],
    }, indent=2, ensure_ascii=False))
    print(f"Saved: {output_path.resolve()}")
    return result


def output_path_for_config(config):
    private_bits = 1 + config.mantissa_bits
    return OUTPUT_PATH.parent / (
        f"{MODEL_SLUG}-w-bfp{private_bits}-a-bie{private_bits}-g{config.block_size}-"
        "p95-p99-grid-mse-rne-fp64exp-calib-train-no-lm-head-s2048.json"
    )


output_paths = []
sweep_summary = []
for mantissa_bits in MANTISSA_BITS_SWEEP:
    config = replace(ABIE, mantissa_bits=mantissa_bits)
    config.validate()
    output_path = output_path_for_config(config)
    print(f"\n=== W-BFP{1 + mantissa_bits}/A-BiE{1 + mantissa_bits}: {output_path.name} ===")
    model, layers = load_and_quantize_model(config)
    try:
        result = evaluate_and_save_width(model, layers, config, output_path)
        output_paths.append(output_path)
        sweep_summary.append({
            "format": f"W-BFP{1 + mantissa_bits}/A-BiE{1 + mantissa_bits}",
            "perplexity": result["perplexity"],
            "delta_perplexity": result["delta_perplexity"],
            "activation_outlier_rate": result["quantization_stats"]["aggregate"]["activation"]["rates"]["outlier_value_rate"]["rate"],
        })
    finally:
        del layers, model
        gc.collect()
        torch.cuda.empty_cache()
if len(output_paths) != len(MANTISSA_BITS_SWEEP):
    raise RuntimeError("W-BFP4/A-BiE4 through W-BFP8/A-BiE8 sweep is incomplete.")


## Sweep summary

The compact view shows PPL and activation outlier rate for each width. Full per-layer
threshold searches and activation outliers-per-block histograms are saved in the JSON files.


In [ ]:
for row in sweep_summary:
    print(
        f"{row['format']}: PPL={row['perplexity']:.6f}, "
        f"delta PPL={row['delta_perplexity']:+.6f}, "
        f"activation outliers={row['activation_outlier_rate'] * 100:.4f}%"
    )


## Download five JSON results

The ZIP contains model-prefixed `w-bfpN-a-bieN` JSON files for N = 4, 5, 6, 7, 8.


In [ ]:
archive_path = OUTPUT_PATH.parent / f"{MODEL_SLUG}-w-bfp4-8-a-bie4-8-g16-p95-p99-results.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in output_paths:
        if not path.is_file():
            raise FileNotFoundError(f"Missing sweep JSON: {path}")
        archive.write(path, arcname=path.name)
print(f"Created: {archive_path.resolve()} ({len(output_paths)} JSON files)")

try:
    from google.colab import files
except ImportError:
    print(f"Not running in Colab. ZIP remains at: {archive_path.resolve()}")
else:
    files.download(str(archive_path))
